# Optional numerical rank check

This notebook checks the ansatz-dependent feature-space dimension, $p=4^n$ for HEA and $p=3^n$ for TPA. The check is diagnostic only: the revised risk implementation fixes $p$ from the ansatz and does not use a floating-point rank threshold to define the theory.


In [6]:
import numpy as np

from QKRR import QuantumKernel

In [7]:
def density_feature_matrix(X, quantum_kernel):
    """Rows are vec(|psi(u)><psi(u)|); their Gram matrix is K."""
    states = np.asarray(
        [quantum_kernel.state_vector(x) for x in X], dtype=complex
    )
    return np.asarray(
        [np.kron(state.conj(), state) for state in states], dtype=complex
    )


def validate_feature_rank(n_qubits, TPA, seed=2026):
    expected_p = 3 ** n_qubits if TPA else 4 ** n_qubits
    N_est = expected_p + max(16, expected_p // 8)
    rng = np.random.default_rng(seed)
    X_est = rng.normal(0, 1, size=(N_est, 2 * n_qubits))

    quantum_kernel = QuantumKernel(N_QUBITS=n_qubits, TPA=TPA)
    features = density_feature_matrix(X_est, quantum_kernel)
    singular_values = np.linalg.svd(features, compute_uv=False)

    # Standard backward-error tolerance, used only for this diagnostic.
    tolerance = max(features.shape) * np.finfo(float).eps * singular_values[0]
    numerical_rank = int(np.count_nonzero(singular_values > tolerance))
    return {
        "ansatz": "TPA" if TPA else "HEA",
        "n_qubits": n_qubits,
        "N_est": N_est,
        "expected_p": expected_p,
        "numerical_rank": numerical_rank,
        "tolerance": tolerance,
        "smallest_retained_singular_value": singular_values[expected_p - 1],
        "largest_discarded_singular_value": (
            singular_values[expected_p]
            if expected_p < singular_values.size
            else 0.0
        ),
    }

In [8]:
configurations = [
    (3, False),
    (3, True),
    (5, False),  # This is the most expensive check (expected p = 1024).
    (5, True),
]

for n_qubits, TPA in configurations:
    result = validate_feature_rank(n_qubits, TPA)
    print(result)
    assert result["numerical_rank"] == result["expected_p"]

{'ansatz': 'HEA', 'n_qubits': 3, 'N_est': 80, 'expected_p': 64, 'numerical_rank': 64, 'tolerance': np.float64(6.43010202343884e-14), 'smallest_retained_singular_value': np.float64(0.06993457088823629), 'largest_discarded_singular_value': 0.0}
{'ansatz': 'TPA', 'n_qubits': 3, 'N_est': 43, 'expected_p': 27, 'numerical_rank': 27, 'tolerance': np.float64(3.552238191588476e-14), 'smallest_retained_singular_value': np.float64(0.19898110694635338), 'largest_discarded_singular_value': np.float64(6.623548928608675e-16)}
{'ansatz': 'HEA', 'n_qubits': 5, 'N_est': 1152, 'expected_p': 1024, 'numerical_rank': 1024, 'tolerance': np.float64(1.6170394220818114e-12), 'smallest_retained_singular_value': np.float64(0.05537307064995625), 'largest_discarded_singular_value': 0.0}
{'ansatz': 'TPA', 'n_qubits': 5, 'N_est': 273, 'expected_p': 243, 'numerical_rank': 243, 'tolerance': np.float64(7.009117050294232e-13), 'smallest_retained_singular_value': np.float64(0.04363623418943136), 'largest_discarded_singula